**Fig. 3, step 3: reconstruct the 11 stacks in 3D with the bootstrap and from-scratch model pairs**

Needs a GPU.

In [ ]:
# Working directory: data/fig3/ under the repository root, where the Fig. 3 deposit folder is downloaded
# (decomposed_2D/, bootstrap_*_models/, domain-specialized_*/, benchmark_outputs*/, metrics_outputs/).
# The 11 ground-truth stacks are read from ../ground_truth/zebrafish/{development,held_out}/.
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # the notebook's own folder, for benchmarking_scripts/
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig3")
print("working directory:", os.getcwd())


# Setups & Imports

In [ ]:
import os
import torch
import multiprocessing as mproc

# IMPORTANT for Jupyter + multiprocessing
mproc.set_start_method("spawn", force=True)

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("Visible GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


CUDA_VISIBLE_DEVICES: 0,1,2
Visible GPU count: 3
0 NVIDIA H200
1 NVIDIA H200
2 NVIDIA H200


In [ ]:
from pathlib import Path
import json
import numpy as np
import tifffile
import torch

import segment3D.usegment3d as uSegment3D
import segment3D.parameters as uSegment3D_params

from benchmarking_scripts.multiview_segmentation_utils import segment_one_view_fullZ
from benchmarking_scripts.notebook_workers import gpu_worker_one_view_notebook

# Specify the Paths to Patches

In [ ]:
# the 7 development stacks
patches_seen = [f'../ground_truth/zebrafish/development/{i}' for i in os.listdir('../ground_truth/zebrafish/development') if 'masks' not in i]
patches_seen.sort()

In [ ]:
# the 4 held-out stacks
patches_unseen = [f'../ground_truth/zebrafish/held_out/{i}' for i in os.listdir('../ground_truth/zebrafish/held_out/') if 'mask' not in i]
patches_unseen.sort()

In [ ]:
# all 11 stacks
patches = patches_seen + patches_unseen
patches

['data/patch-1.tif',
 'data/patch-2.tif',
 'data/patch-3.tif',
 'data/patch-4.tif',
 'data/patch-5.tif',
 'data/patch-7.tif',
 'data/patch-8.tif',
 'data_holdout/patch-10.tif',
 'data_holdout/patch-14.tif',
 'data_holdout/patch-6.tif',
 'data_holdout/patch-9.tif']

# Specify the Paths to Model Pairs

In [ ]:
# Bootstrap model pairs (rounds 0, 4, 9)
bootstrap_axial_models = os.listdir('bootstrap_axial_models/')
bootstrap_axial_models.sort()
bootstrap_axial_models = [f'bootstrap_axial_models/{i}' for i in bootstrap_axial_models]

bootstrap_lateral_models = os.listdir('bootstrap_lateral_models/')
bootstrap_lateral_models.sort()
bootstrap_lateral_models = [f'bootstrap_lateral_models/{i}' for i in bootstrap_lateral_models]

bootstrap_model_pairs_paths = list(zip(bootstrap_lateral_models, bootstrap_axial_models))
bootstrap_model_pairs_paths

[('bootstrap_lateral_models/spatial_model_round0',
  'bootstrap_axial_models/axial_model_round0'),
 ('bootstrap_lateral_models/spatial_model_round4',
  'bootstrap_axial_models/axial_model_round4'),
 ('bootstrap_lateral_models/spatial_model_round9',
  'bootstrap_axial_models/axial_model_round9')]

In [ ]:
# From-scratch (ceiling) model pairs
axial_rescale_true = [f'domain-specialized_axial_models/rescale_true/models/{i}' 
                      for i in os.listdir('domain-specialized_axial_models/rescale_true/models/') if '.npy' not in i]
axial_rescale_true.sort()
lateral_rescale_true = [f'domain-specialized_lateral_models/rescale_true/models/{i}' 
                        for i in os.listdir('domain-specialized_lateral_models/rescale_true/models/') if '.npy' not in i]
lateral_rescale_true.sort()

pairs_rescale_true_paths = list(zip(lateral_rescale_true, axial_rescale_true))
pairs_rescale_true_paths

[('domain-specialized_lateral_models/rescale_true/models/domain-specialized_lateral_500epoch_scratch_7patch_lenient_rescale_True',
  'domain-specialized_axial_models/rescale_true/models/domain-specialized_axial_500epoch_scratch_7patch_lenient_rescale_True'),
 ('domain-specialized_lateral_models/rescale_true/models/domain-specialized_lateral_500epoch_scratch_7patch_strict_rescale_True',
  'domain-specialized_axial_models/rescale_true/models/domain-specialized_axial_500epoch_scratch_7patch_strict_rescale_True'),
 ('domain-specialized_lateral_models/rescale_true/models/domain-specialized_lateral_500epoch_scratch_lenient_rescale_True',
  'domain-specialized_axial_models/rescale_true/models/domain-specialized_axial_500epoch_scratch_lenient_rescale_True'),
 ('domain-specialized_lateral_models/rescale_true/models/domain-specialized_lateral_500epoch_scratch_strict_rescale_True',
  'domain-specialized_axial_models/rescale_true/models/domain-specialized_axial_500epoch_scratch_strict_rescale_True

In [ ]:
# final list of models to evaluate
from pathlib import Path

model_pairs = []

# --------------------------------------------------
# 1. Bootstrap models
# --------------------------------------------------
for i, (lat, ax) in enumerate(bootstrap_model_pairs_paths):
    pair_id = f"bootstrap_models{i}"
    model_pairs.append({
        "id": pair_id,
        "lateral": lat,
        "axial": ax,
        "rescale": True,
        "group": "bootstrap",
    })
    
# --------------------------------------------------
# 2. From-scratch (ceiling) models
# --------------------------------------------------
for lat, ax in pairs_rescale_true_paths:
    name = Path(lat).name
    pair_id = name.replace("domain-specialized_lateral_", "")
    model_pairs.append({
        "id": pair_id,
        "lateral": lat,
        "axial": ax,
        "rescale": True,
        "group": "domain_rescale_true",
    })

# --------------------------------------------------
# 3. Sanity check
# --------------------------------------------------
print(f"Total model pairs: {len(model_pairs)}\n")

for mp in model_pairs:
    print(
        f"{mp['id']:<45} | "
        f"group={mp['group']:<20} | "
        f"rescale={mp['rescale']}"
    )


# Function to Perform Reconstruction

In [ ]:
def run_one_patch_one_pair_notebook(
    patch_path,
    model_pair,
    out_root,
    base_segment_params,
    reconstruction_params,
    reconstruction_post_params,
    gpu_triplet=(0,1,2),
    verbose=True,
):
    """Reconstruct one stack with one lateral/axial model pair and score nothing yet.

    Runs `gpu_worker_one_view_notebook` for the XY (lateral model), XZ and YZ (axial model)
    views in three processes, one GPU each, collects the per-view probability and flow maps,
    fuses them with u-Segment3D's direct method (`reconstruction_params`), post-processes the
    3D labels (`reconstruction_post_params`), and writes
    `<out_root>/<stack>/<model_id>_mask.tif` (uint32) plus `<model_id>_meta.json`
    (stack, model id and group, rescale flag, the two checkpoint paths, mask path).
    Returns the meta dict.
    """
    import multiprocessing as mp

    segment_params = dict(base_segment_params)
    segment_params["rescale"] = model_pair["rescale"]

    q = mp.Queue()
    procs = []

    jobs = [
        ("xy", model_pair["lateral"], gpu_triplet[0]),
        ("xz", model_pair["axial"],   gpu_triplet[1]),
        ("yz", model_pair["axial"],   gpu_triplet[2]),
    ]

    for view, model_path, gpu in jobs:
        p = mp.Process(
            target=gpu_worker_one_view_notebook,
            kwargs=dict(
                patch_path=patch_path,
                model_path=model_path,
                view=view,
                segment_params=segment_params,
                gpu_id=gpu,
                out_queue=q,
                verbose=verbose,
            ),
        )
        p.start()
        procs.append(p)

    results = {}
    for _ in range(3):
        item = q.get()
        if "error" in item:
            for p in procs:                 # a worker still holding results on the queue
                p.terminate()               # would never exit: stop it before joining
                p.join()
            raise RuntimeError(item)
        results[item["view"]] = item

    for p in procs:
        p.join()
    
    seg3D, (prob3D, grad3D) = uSegment3D.aggregate_2D_to_3D_segmentation_direct_method(
        probs=[
            results["xy"]["probs"],
            results["xz"]["probs"],
            results["yz"]["probs"],
        ],
        gradients=[
            results["xy"]["flows"],
            results["xz"]["flows"],
            results["yz"]["flows"],
        ],
        params=reconstruction_params,
    )
    
    seg3D, _ = uSegment3D.postprocess_3D_cell_segmentation(
        seg3D,
        aggregation_params=reconstruction_params,
        postprocess_params=reconstruction_post_params,
        cell_gradients=grad3D,
        savefolder=None,
        basename=None,
    )

    out_dir = Path(out_root) / Path(patch_path).stem
    out_dir.mkdir(parents=True, exist_ok=True)

    mask_path = out_dir / f"{model_pair['id']}_mask.tif"
    tifffile.imwrite(mask_path, seg3D.astype(np.uint32), bigtiff=True)

    # save the metadata
    meta = {
        "patch": Path(patch_path).stem,
        "patch_path": patch_path,
    
        "model_id": model_pair["id"],
        "model_group": model_pair["group"],
        "rescale": model_pair["rescale"],
    
        "lateral_model": model_pair["lateral"],
        "axial_model": model_pair["axial"],
    
        "mask_path": str(mask_path),
    }
    
    meta_path = out_dir / f"{model_pair['id']}_meta.json"
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)


    return meta


# Set Segmentation & Reconstruction Parameters

In [ ]:
# Cellpose prediction parameters (Table S6)
base_segment_params = {
    "channels": [2, 1],
    "normalize": True,
    "cellprob_threshold": 0.0,
    "flow_threshold": 0.4,
    "batch_size": 1024,
}

# define the reconstructions parameters to be used for the fusion method
direct_aggregation_params = uSegment3D_params.get_2D_to_3D_aggregation_params()

# the probability maps are Cellpose cell probabilities
direct_aggregation_params['combine_cell_probs']['cellpose_prob_mask'] = True 
direct_aggregation_params['combine_cell_probs']['ksize'] = 25
direct_aggregation_params['combine_cell_probs']['alpha'] = 0.5
direct_aggregation_params['combine_cell_probs']['smooth_sigma'] = 0.8

direct_aggregation_params['combine_cell_gradients']['ksize'] = 25
direct_aggregation_params['combine_cell_gradients']['smooth_sigma'] = 1.5
direct_aggregation_params['combine_cell_gradients']['post_sigma'] = 0.5

# post process autoinfered binary from probability masks
direct_aggregation_params['postprocess_binary']['binary_fill_holes'] = True
direct_aggregation_params['postprocess_binary']['remove_snall_objects'] = True
direct_aggregation_params['postprocess_binary']['min_size'] = 250

# Gradient descent vector field integration
direct_aggregation_params['gradient_descent']['n_iter'] = 50
direct_aggregation_params['gradient_descent']['momenta'] = 0.98
direct_aggregation_params['gradient_descent']['gradient_decay'] = 0.001

direct_aggregation_params['gradient_descent']['debug_viz'] = False
direct_aggregation_params['gradient_descent']['do_mp'] = True
direct_aggregation_params['gradient_descent']['renorm_gradient'] = False

post_process_params = uSegment3D_params.get_postprocess_segmentation_params()
post_process_params['flow_consistency']['do_flow_remove'] = False  # flow-consistency removal off
post_process_params['flow_consistency']['flow_threshold'] = 1.0
post_process_params['flow_consistency']['n_cpu'] = 24
post_process_params['flow_consistency']['dtform_method'] = 'cellpose_improve'

In [ ]:
# Reconstruct every stack with every model pair
for patch in patches:
    for mpair in model_pairs:
        print(f'running {mpair} on patch:')
        run_one_patch_one_pair_notebook(
            patch_path=patch,
            model_pair=mpair,
            out_root="benchmark_outputs",
            base_segment_params=base_segment_params,
            reconstruction_params=direct_aggregation_params,
            reconstruction_post_params=post_process_params,
            verbose=True,
        )
